# Raw EEG Signal

**Dataset**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 7

---

## Overview

This notebook plots the raw signal recorded from the four scalp electrodes. This is the signal before any processing or filtering — the starting point for all analysis.

What to observe in the raw signal:
- Large slow waves from eye movement and respiration
- Smaller faster oscillations reflecting actual brain activity
- Differences between channels based on electrode location


## 1. Install dependencies

In [ ]:
!pip install scipy numpy plotly wfdb

## 2. Clone repository and download subject 7

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')

In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 7

## 3. Load raw signal

We load subject 7, experiment 1, session 2. The loader returns four channels: P4, Cz, F8, T7.

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=7, experiment=1, session=2
)
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal shape: {eeg_data.shape}')
print(f'Duration: {eeg_data.shape[0]/fs:.1f} seconds')

## 4. Plot raw signal for all four channels

We display the first ~25 seconds of each channel in a separate subplot. This helps see differences between channels and spot obvious artifacts.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, eeg_data.shape[0])
t_sec = timestamps[:n_plot] / 1000.0

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    vertical_spacing=0.04,
                    subplot_titles=[f'Channel {ch}' for ch in ch_names])

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for i, (ch, color) in enumerate(zip(ch_names, colors)):
    fig.add_trace(go.Scatter(x=t_sec, y=eeg_data[:n_plot, i],
                            mode='lines', name=ch,
                            line=dict(color=color, width=0.8)),
                  row=i+1, col=1)

fig.update_layout(height=800, width=1000,
                  title_text='Raw EEG Signal - Subject 7 (first 25 s)',
                  showlegend=False)
fig.update_xaxes(title_text='Time (s)', row=4, col=1)
fig.show()

## 5. Summary

- The raw signal contains a mix of brain activity and artifacts
- Each channel reflects activity from a different brain region
- Obvious artifacts like eye movement appear strongly in frontal channels
- This signal is the starting point for all processing steps in later chapters